## Setup

In [1]:
import json
import os

import pandas as pd

In [2]:
#with open('config.json') as configfile:
#    cfg = json.load(configfile)
#os.environ['HF_TOKEN'] = cfg['HF_TOKEN']

In [ ]:
# Define our datasets

# Dataset 1: AzharAli05/Resume-Screening-Dataset
dataset_1_url = "hf://datasets/AzharAli05/Resume-Screening-Dataset/dataset.csv"

# Dataset 2: MikePfunk28/resume-training-dataset
#dataset_2_url = "hf://datasets/MikePfunk28/resume-training-dataset/training_data.jsonl"
# HF_TOKEN necessary``

# Dataset 3: datasetmaster/resumes
#dataset_3_url = "hf://dataset/datasetmaster/resumes/master_resumes.jsonl"
# doesn't seem to work

## Dataset Prep

### Dataset 1
https://huggingface.co/datasets/AzharAli05/Resume-Screening-Dataset

In [ ]:
df_1_raw = pd.read_csv(dataset_1_url)
print(df_1_raw.shape)
df_1_raw.head()

(10174, 5)


,Role,Resume,Decision,Reason_for_decision,Job_Description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [5]:
# I'll process the whole dataset down to a subset with a known format
# we don't mind losing a bunch of resumes, we don't need that many
df_1 = df_1_raw.copy()
df_1.columns = ['role', 'resume', 'decision', 'reasoning', 'description']
df_1['decision'] = (df_1['decision'] == "select")
print(df_1.shape)
df_1.head()

(10174, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,True,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [6]:
# Only extract resumes with a known format
# "Here" as a prefix indicates that this a LLM-generated resume ("Here's a professional resume for...")
# while "Available upon request" indicates that this resume has a references section, and so we know where it ends
prefix = "Here"
suffix = "Available upon request."
df_1 = df_1[df_1['resume'].str.startswith(prefix)]
df_1 = df_1[df_1['resume'].str.contains(suffix)]
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,"Here's a sample resume for Jose Hall, a skille...",False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [7]:
# Remove the "Here's a resume!" prefix
df_1['resume'] = df_1['resume'].str.split(':\n\n', n=1).str[1]
df_1 = df_1[~df_1['resume'].isna()]
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Patrick Mcclain\nHuman Resources Specialist\n\...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [8]:
# and remove anything after the references (this is usually LLM commentary)
df_1['resume'] = df_1['resume'].str.rsplit(suffix, n=1).str[0] + suffix
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Patrick Mcclain\nHuman Resources Specialist\n\...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [9]:
# Explore: how many unique positions are there?
df_1_resume_counts = df_1.groupby('description')['resume'].count().reset_index()
df_1_resume_counts.columns = ['description', 'resume_count']
print(df_1_resume_counts.shape)
df_1_resume_counts

(1714, 2)


,description,resume_count
0,**Job Title: Data Scientist**\n\n**Job Summary...,1
1,**Job Title: Database Administrator**\n\n**Job...,1
2,**Job Title: HR Specialist**\n\n**Job Summary:...,1
3,**Job Title: UI/UX Designer**\n\n**Job Summary...,1
4,**Job Title:** AI Engineer\n\n**Job Summary:**...,1
...,...,...
1709,We're seeking a talented UI Engineer to work o...,7
1710,We're seeking a talented UI Engineer to work o...,5
1711,We're seeking a talented UX Designer to work o...,5
1712,We're seeking a talented UX Designer to work o...,5


In [10]:
# and of these positions, how many have both accepted and rejected resumes?
df_1_dual_decision = df_1.groupby('description')['decision'].nunique().reset_index()
df_1_dual_decision.columns = ['description', 'decision_types']
df_1_dual_decision = pd.merge(
    df_1_dual_decision, df_1_resume_counts,
    on='description', how='left'
)
df_1_dual_decision = df_1_dual_decision[df_1_dual_decision['decision_types'] == 2]
print(df_1_dual_decision.shape)
df_1_dual_decision

(864, 3)


,description,decision_types,resume_count
66,"As a AI Researcher, you will play a pivotal ro...",2,8
67,"As a AI Researcher, you will play a pivotal ro...",2,6
68,"As a AI Researcher, you'll lead the design and...",2,4
69,"As a AI Researcher, you'll lead the design and...",2,5
70,"As a AI Researcher, you'll lead the design and...",2,4
...,...,...,...
1708,We're seeking a talented UI Engineer to work o...,2,2
1709,We're seeking a talented UI Engineer to work o...,2,7
1710,We're seeking a talented UI Engineer to work o...,2,5
1712,We're seeking a talented UX Designer to work o...,2,5


In [11]:
# Narrow the dataset down to positions with both accepts and rejects
df_1 = df_1[df_1['description'].isin(df_1_dual_decision['description'])]
df_1 = df_1.reset_index(drop=True)
print(df_1.shape)
df_1.head()

(4861, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
2,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
3,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...
4,Cloud Engineer,Jessica Hall\nCloud Engineer\n\nContact Inform...,False,Needs improvement in machine learning algorithms.,We're seeking a talented Cloud Engineer to wor...


In [12]:
# Sample a job opening from the entire dataset, and show all its applications
random_state = 0
df_1_subset = df_1[df_1['description'] == df_1['description'].sample(random_state = random_state).item()]
print(df_1_subset['description'].iloc[0])
df_1_subset

As a System Administrator, you will play a pivotal role in shaping the future of e-commerce.


,role,resume,decision,reasoning,description
311,System Administrator,Christina Davis\nSystem Administrator\n\nConta...,False,Needs improvement in machine learning algorithms.,"As a System Administrator, you will play a piv..."
915,System Administrator,Mary Johnson\nSystem Administrator\n\nContact ...,False,Insufficient system design expertise for senio...,"As a System Administrator, you will play a piv..."
2133,System Administrator,John Irwin\nSystem Administrator\n\nContact In...,True,Impressive leadership and communication abilit...,"As a System Administrator, you will play a piv..."
3202,System Administrator,Travis Dawson\nSystem Administrator\n\nContact...,True,Perfectly aligned with data engineering needs.,"As a System Administrator, you will play a piv..."
3827,System Administrator,Marvin Bates\nSystem Administrator\n\nContact ...,True,Excellent full-stack development experience.,"As a System Administrator, you will play a piv..."
4105,System Administrator,Matthew Moreno\nSystem Administrator Candidate...,False,Lacked leadership skills for a senior position.,"As a System Administrator, you will play a piv..."
4258,System Administrator,Jerry Nelson\nSystem Administrator Candidate\n...,True,Strong technical skills in AI and ML.,"As a System Administrator, you will play a piv..."


In [13]:
# and sample a random resume 
print(df_1_subset['resume'].sample().item())

Matthew Moreno
System Administrator Candidate

Contact Information:

* Phone: (123) 456-7890
* Email: [mmoreno@email.com](mailto:mmoreno@email.com)
* LinkedIn: linkedin.com/in/matthewmoreno

Professional Summary:
Highly motivated and experienced System Administrator with a strong background in Windows Server, Active Directory, Virtualization, and System Monitoring. Proven track record of successfully implementing and managing complex IT systems, ensuring high uptime and performance. Skilled in troubleshooting, problem-solving, and collaboration. Seeking a challenging role as a System Administrator where I can utilize my expertise to drive business growth and success.

Technical Skills:

* Windows Server (2012, 2016, 2019)
* Active Directory (AD DS, AD LDS, AD FS)
* Virtualization (VMware vSphere, Hyper-V)
* System Monitoring (SCOM, Nagios, SolarWinds)
* Network Protocols (TCP/IP, DNS, DHCP, HTTP)
* Scripting Languages (PowerShell, Batch)
* Operating Systems (Windows, Linux)

Profession

## Final Datasets

In [14]:
# Dataset 1
df_1.sample(10, random_state=0)

,role,resume,decision,reasoning,description
311,System Administrator,Christina Davis\nSystem Administrator\n\nConta...,False,Needs improvement in machine learning algorithms.,"As a System Administrator, you will play a piv..."
154,Product Manager,Anthony Mckenzie\nContact Information:\n\n* Ad...,False,Lacks hands-on experience with cloud platforms.,"As a Product Manager, you will play a pivotal ..."
2027,Product Manager,Sheila Nguyen\nContact Information:\n\n* Addre...,True,Impressive leadership and communication abilit...,Looking for an experienced Product Manager to ...
1687,Cloud Architect,Jonathan Cook\nCloud Architect\n\nContact Info...,False,Insufficient system design expertise for senio...,Be part of a passionate team at the forefront ...
1519,IT Support Specialist,Richard Fuller\nContact Information:\n\n* Phon...,False,Insufficient system design expertise for senio...,We are looking for an experienced IT Support S...
2044,Data Analyst,Katherine Tyler\nContact Information:\n\n* Ema...,True,Strong technical skills in AI and ML.,We are looking for an experienced Data Analyst...
4100,AI Researcher,Crystal Park\nContact Information:\n\n* Email:...,False,Needs improvement in machine learning algorithms.,Be part of a passionate team at the forefront ...
4684,QA Engineer,Raymond Greer\nQA Engineer\n\nContact Informat...,False,No experience in back-end development.,Help us build the next-generation products as ...
1881,DevOps Engineer,Lynn Hancock\nContact Information:\n\n* Phone:...,True,Solid experience in machine learning and AI.,Help us build the next-generation products as ...
1108,Mobile App Developer,Charles Phillips\nMobile App Developer\n\nCont...,False,Insufficient system design expertise for senio...,We are looking for an experienced Mobile App D...
